# Synode — Notebook 01
## Repository Dataset Builder

Build a reproducible dataset from fixed GitHub repositories
for code intelligence, risk prediction, and change-impact prediction.


In [ ]:
!pip -q install GitPython pandas tqdm python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.0/221.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm
from git import Repo

In [ ]:
BASE_DIR = Path("/content/synode_dataset")

REPO_DIR = BASE_DIR / "repositories"
DATA_DIR = BASE_DIR / "data"

REPO_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Repository directory:", REPO_DIR)
print("Data directory:", DATA_DIR)

Base directory: /content/synode_dataset
Repository directory: /content/synode_dataset/repositories
Data directory: /content/synode_dataset/data


In [ ]:
REPOSITORIES = [
    {
        "name": "repository_1",
        "url": "https://github.com/pallets/flask.git"
    },
    {
        "name": "repository_2",
        "url": "https://github.com/fastapi/fastapi.git"
    },
    {
        "name": "repository_3",
        "url": "https://github.com/encode/django-rest-framework.git"
    }
]

In [ ]:
def clone_repository(name, url):
    repo_path = REPO_DIR / name

    if repo_path.exists():
        print(f"{name} already exists. Skipping.")
        return repo_path

    print(f"Cloning {name}...")

    Repo.clone_from(
        url,
        repo_path
    )

    print(f"Cloned to: {repo_path}")

    return repo_path

In [ ]:
for repo_info in REPOSITORIES:
    clone_repository(
        repo_info["name"],
        repo_info["url"]
    )

Cloning repository_1...
Cloned to: /content/synode_dataset/repositories/repository_1
Cloning repository_2...
Cloned to: /content/synode_dataset/repositories/repository_2
Cloning repository_3...
Cloned to: /content/synode_dataset/repositories/repository_3


In [ ]:
for repo_info in REPOSITORIES:
    repo_path = REPO_DIR / repo_info["name"]

    if repo_path.exists():
        repo = Repo(repo_path)

        print("=" * 60)
        print("Repository:", repo_info["name"])
        print("Current commit:", repo.head.commit.hexsha)
        print("Branch:", repo.active_branch.name)

Repository: repository_1
Current commit: d318b683471101618febed18996405ad26462110
Branch: main
Repository: repository_2
Current commit: c3f316b7e814667e8ee81e03a7330d00ee61e45c
Branch: master
Repository: repository_3
Current commit: 751a19fe1b237beca9af7d587fce55d3e09d3741
Branch: main


In [ ]:
repository_records = []

for repo_info in REPOSITORIES:
    repo_path = REPO_DIR / repo_info["name"]
    repo = Repo(repo_path)

    commit = repo.head.commit

    repository_records.append({
        "repository_name": repo_info["name"],
        "repository_url": repo_info["url"],
        "commit_sha": commit.hexsha,
        "branch": repo.active_branch.name,
        "snapshot_created_at": datetime.utcnow().isoformat()
    })

repositories_df = pd.DataFrame(repository_records)

repositories_df

/tmp/ipykernel_4567/1879789107.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "snapshot_created_at": datetime.utcnow().isoformat()
/tmp/ipykernel_4567/1879789107.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "snapshot_created_at": datetime.utcnow().isoformat()
/tmp/ipykernel_4567/1879789107.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "snapshot_created_at": datetime.utcnow().isoformat()


,repository_name,repository_url,commit_sha,branch,snapshot_created_at
0,repository_1,https://github.com/pallets/flask.git,d318b683471101618febed18996405ad26462110,main,2026-08-23T19:12:13.648946
1,repository_2,https://github.com/fastapi/fastapi.git,c3f316b7e814667e8ee81e03a7330d00ee61e45c,master,2026-08-23T19:12:13.652193
2,repository_3,https://github.com/encode/django-rest-framewor...,751a19fe1b237beca9af7d587fce55d3e09d3741,main,2026-08-23T19:12:13.654915


In [ ]:
repositories_df.to_csv(
    DATA_DIR / "repositories.csv",
    index=False
)

In [ ]:
IGNORED_DIRECTORIES = {
    ".git",
    "node_modules",
    "venv",
    ".venv",
    "__pycache__",
    "dist",
    "build",
    ".next",
    "coverage",
    ".pytest_cache",
    ".mypy_cache",
}

IGNORED_EXTENSIONS = {
    ".pyc",
    ".pyo",
    ".so",
    ".dll",
    ".exe",
    ".bin",
    ".zip",
    ".tar",
    ".gz",
    ".jpg",
    ".jpeg",
    ".png",
    ".gif",
    ".mp4",
    ".mp3",
}

In [ ]:
EXTENSION_TO_LANGUAGE = {
    ".py": "Python",
    ".js": "JavaScript",
    ".jsx": "JavaScript",
    ".ts": "TypeScript",
    ".tsx": "TypeScript",
    ".java": "Java",
    ".go": "Go",
    ".rs": "Rust",
    ".cpp": "C++",
    ".c": "C",
    ".h": "C/C++",
    ".hpp": "C++",
    ".md": "Markdown",
    ".json": "JSON",
    ".yml": "YAML",
    ".yaml": "YAML",
}

In [ ]:
def should_ignore(path):
    parts = set(path.parts)

    if parts.intersection(IGNORED_DIRECTORIES):
        return True

    if path.suffix.lower() in IGNORED_EXTENSIONS:
        return True

    return False

In [ ]:
def scan_repository(repo_name, repo_path):

    records = []

    for path in repo_path.rglob("*"):

        if not path.is_file():
            continue

        relative_path = path.relative_to(repo_path)

        if should_ignore(relative_path):
            continue

        language = EXTENSION_TO_LANGUAGE.get(
            path.suffix.lower()
        )

        if language is None:
            continue

        try:
            size_bytes = path.stat().st_size

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                content = f.read()

            loc = len(content.splitlines())

        except Exception:
            continue

        records.append({
            "repository_name": repo_name,
            "path": str(relative_path),
            "language": language,
            "extension": path.suffix.lower(),
            "size_bytes": size_bytes,
            "loc": loc
        })

    return records

In [ ]:
all_file_records = []

for repo_info in REPOSITORIES:

    repo_name = repo_info["name"]
    repo_path = REPO_DIR / repo_name

    print(f"Scanning {repo_name}...")

    records = scan_repository(
        repo_name,
        repo_path
    )

    all_file_records.extend(records)

files_df = pd.DataFrame(all_file_records)

files_df.head()

Scanning repository_1...
Scanning repository_2...
Scanning repository_3...


,repository_name,path,language,extension,size_bytes,loc
0,repository_1,README.md,Markdown,.md,1639,53
1,repository_1,.readthedocs.yaml,YAML,.yaml,242,10
2,repository_1,.pre-commit-config.yaml,YAML,.yaml,844,24
3,repository_1,.devcontainer/devcontainer.json,JSON,.json,434,17
4,repository_1,tests/test_blueprints.py,Python,.py,31287,1115


In [ ]:
print("Total files:", len(files_df))

print("\nLanguages:")
print(files_df["language"].value_counts())

print("\nRepositories:")
print(files_df["repository_name"].value_counts())

print("\nTotal LOC:")
print(files_df["loc"].sum())

Total files: 3227

Languages:
language
Markdown      1775
Python        1375
YAML            63
JavaScript      12
JSON             2
Name: count, dtype: int64

Repositories:
repository_name
repository_2    2876
repository_3     252
repository_1      99
Name: count, dtype: int64

Total LOC:
448947


In [ ]:
def is_test_file(path):

    name = Path(path).name.lower()

    return (
        name.startswith("test_")
        or name.endswith("_test.py")
        or ".test." in name
        or ".spec." in name
        or "/tests/" in path.lower()
        or "\\tests\\" in path.lower()
    )

In [ ]:
files_df["is_test"] = files_df["path"].apply(is_test_file)

In [ ]:
files_df["is_test"].value_counts()

,count
is_test,
False,2601
True,626


In [ ]:
def get_module(path):

    path_obj = Path(path)

    if len(path_obj.parts) <= 1:
        return "root"

    return path_obj.parts[0]

In [ ]:
files_df["module"] = files_df["path"].apply(get_module)

In [ ]:
files_df.to_csv(
    DATA_DIR / "files.csv",
    index=False
)

print("Saved:", DATA_DIR / "files.csv")

Saved: /content/synode_dataset/data/files.csv


In [ ]:
def extract_commits(repo_name, repo_path):

    repo = Repo(repo_path)

    records = []

    for commit in repo.iter_commits():

        records.append({
            "repository_name": repo_name,
            "sha": commit.hexsha,
            "author": commit.author.name,
            "author_email": commit.author.email,
            "timestamp": datetime.fromtimestamp(
                commit.committed_date
            ).isoformat(),
            "message": commit.message.strip()
        })

    return records

In [ ]:
all_commit_records = []

for repo_info in REPOSITORIES:

    print(
        f"Extracting commits from {repo_info['name']}..."
    )

    records = extract_commits(
        repo_info["name"],
        REPO_DIR / repo_info["name"]
    )

    all_commit_records.extend(records)

commits_df = pd.DataFrame(all_commit_records)

print("Total commits:", len(commits_df))

Extracting commits from repository_1...
Extracting commits from repository_2...
Extracting commits from repository_3...
Total commits: 22290


In [ ]:
commits_df.to_csv(
    DATA_DIR / "commits.csv",
    index=False
)

In [ ]:
def extract_changes(repo_name, repo_path):

    repo = Repo(repo_path)

    records = []

    for commit in repo.iter_commits():

        if not commit.parents:
            continue

        parent = commit.parents[0]

        try:
            diffs = parent.diff(
                commit,
                create_patch=True
            )
        except Exception:
            continue

        for diff in diffs:

            path = (
                diff.b_path
                or diff.a_path
                or ""
            )

            if not path:
                continue

            try:
                added = diff.diff.count(b"+")
                removed = diff.diff.count(b"-")
            except Exception:
                added = 0
                removed = 0

            records.append({
                "repository_name": repo_name,
                "commit_sha": commit.hexsha,
                "file_path": path,
                "lines_added": added,
                "lines_removed": removed
            })

    return records

In [ ]:
all_change_records = []

for repo_info in REPOSITORIES:

    print(
        f"Extracting changes from {repo_info['name']}..."
    )

    records = extract_changes(
        repo_info["name"],
        REPO_DIR / repo_info["name"]
    )

    all_change_records.extend(records)

changes_df = pd.DataFrame(all_change_records)

print("Total change events:", len(changes_df))

Extracting changes from repository_1...
Extracting changes from repository_2...
Extracting changes from repository_3...
Total change events: 79679


In [ ]:
changes_df.to_csv(
    DATA_DIR / "changes.csv",
    index=False
)

In [ ]:
summary = {
    "repositories": len(repositories_df),
    "files": len(files_df),
    "commits": len(commits_df),
    "change_events": len(changes_df),
    "total_loc": int(files_df["loc"].sum()),
    "languages": files_df["language"].value_counts().to_dict(),
    "generated_at": datetime.utcnow().isoformat()
}

with open(
    DATA_DIR / "dataset_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=4
    )

summary

/tmp/ipykernel_4567/2772249872.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat()


{'repositories': 3,
 'files': 3227,
 'commits': 22290,
 'change_events': 79679,
 'total_loc': 448947,
 'languages': {'Markdown': 1775,
  'Python': 1375,
  'YAML': 63,
  'JavaScript': 12,
  'JSON': 2},
 'generated_at': '2026-08-23T19:20:24.108301'}